#Predicitive Maintenance


## **Notebook Purpose**
The purpose of this notebook is to train and evaluate a Convolutional Neural Network (CNN) capable of identifying mechanical faults in industrial machinery using only acoustic data. By converting raw machine audio into Log-Mel-Spectrograms, the model learns the physical acoustic signatures of both healthy ("Normal") and failing ("Abnormal") equipment.

In a real-world industrial setting, this type of model prevents catastrophic equipment failure, reduces downtime, and cuts maintenance costs by alerting engineers to microscopic mechanical changes before a machine completely breaks down.

### **Pipeline Execution Flow**
This notebook acts as the cloud execution environment for the project, utilizing a cloud GPU to handle heavy computational loads. The workflow is split into four automated phases:
1. **Environment Setup:** Cloning the pipeline architecture from GitHub and installing dependencies.
2. **Data Ingestion:** Fetching the raw `.wav` files (e.g., pump audio) and preprocessing them into `.npy` spectrogram arrays.
3. **Hyperparameter Optimization:** Systematically training multiple variations of the model to find the optimal architecture.
4. **Artifact Export:** Compiling the trained `.pth` weights, evaluation metrics, and visual explanations (Grad-CAM heatmaps, ROC Curves) into downloadable archives.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 2. Start fresh and clone the newly updated code
%cd /content
!rm -rf Predictive-Maintenance-Audio-CNN
!git clone https://github.com/CreatorRa/Predictive-Maintenance-Audio-CNN.git
%cd Predictive-Maintenance-Audio-CNN

# 3. Install the environment (Verbose mode as requested)
!pip install -r requirements.txt

# 4. Fetch and extract the audio data
!cp "/content/drive/MyDrive/6_dB_pump.zip" .
!unzip 6_dB_pump.zip

# 5. Route the data to the correct folders using the wildcard fix
!mkdir -p data/raw
!mv */"Normal (Master)" data/raw/normal
!mv */"Abnormal (Master)" data/raw/abnormal
!rm 6_dB_pump.zip

/content
Cloning into 'Predictive-Maintenance-Audio-CNN'...
remote: Enumerating objects: 122, done.
remote: Counting objects: 100% (122/122), done.
remote: Compressing objects: 100% (105/105), done.
remote: Total 122 (delta 35), reused 96 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (122/122), 13.13 MiB | 38.40 MiB/s, done.
Resolving deltas: 100% (35/35), done.
/content/Predictive-Maintenance-Audio-CNN
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 84.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 81.1 MB/s eta 0:00:00
  Created wheel for grad-cam: filename=grad_cam-1.5.5-py3-none-any.whl size=44286 sha256=961f79b0da0b2c2f41950e7689de0d4635657d71e55ab33b9da8c45473fbea2e
  Stored in directory: /root/.cache/pip/wheels/fb/3b/09/2afc520f3d69bc26ae6bd87416759c820a3f7d05c1a077bbf6
Successfully built grad-cam


In [3]:
#Run the master orchestrator!
!python run_pipeline.py

  PREDICTIVE MAINTENANCE AUDIO CNN — FULL PIPELINE
  Stages to run: 7

  STAGE: Test Suite (pytest)
  CMD:   /usr/bin/python3 -m pytest tests/ -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/Predictive-Maintenance-Audio-CNN
plugins: anyio-4.13.0, langsmith-0.7.32, typeguard-4.5.1
collected 16 items                                                             

tests/test_dataset.py::test_class_weights_heavily_penalize_minority_class PASSED [  6%]
tests/test_dataset.py::test_class_weights_balanced_input_produces_unit_weights PASSED [ 12%]
tests/test_dataset.py::test_class_weights_dtype_is_float32 PASSED        [ 18%]
tests/test_dataset.py::test_stratified_split_preserves_class_ratio_in_every_subset PASSED [ 25%]
tests/test_dataset.py::test_stratified_split_ratios_are_70_15_15 PASSED  [ 31%]
tests/test_dataset.py::test_stratified_s

In [ ]:
import shutil
from google.colab import files

# Zip the visualizations folder
shutil.make_archive('final_visualizations', 'zip', '/content/Predictive-Maintenance-Audio-CNN/docs', 'visualizations')

# Zip the models folder
shutil.make_archive('final_model', 'zip', '/content/Predictive-Maintenance-Audio-CNN/models')

# Trigger the downloads to your local computer
files.download('final_visualizations.zip')
files.download('final_model.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Hyperparameter Optimization (`tune.py`)

Finding the perfect neural network architecture requires rigorous, systematic testing rather than random guessing. The cell below runs `tune.py`, which acts as the master orchestrator for a **Cartesian Grid Search**.

### **What this script is doing:**
It automatically trains and evaluates multiple completely distinct models by testing every possible combination of the following hyperparameters:
* **Learning Rate:** Controls how aggressively the optimizer updates the model's weights (e.g., `1e-3` vs `5e-4`).
* **Batch Size:** The number of spectrograms processed simultaneously, which influences the mathematical stability of the training loop.
* **Base Filters:** Controls the "width" and feature-detection capacity of the CNN.

### **Engineering Design:**
To prevent the GPU from running out of memory (VRAM) during these consecutive training runs, `tune.py` executes each experiment as an isolated child process using Python's `subprocess` module. Once an experiment finishes, the VRAM is wiped perfectly clean for the next one.

As the models finish training, the script automatically tests them against an isolated hold-out set, records their **F1-Score** and **ROC-AUC** metrics to a master CSV tracker, and generates a visual summary graph of the entire sweep.

In [4]:
#Run the Hypertuning scripts
!python tune.py

HPO GRID SEARCH — 8 combinations
              lr: [0.001, 0.0005]
      batch_size: [16, 32]
    base_filters: [16, 32]

[1/8] run_lr1em03_bs16_bf16
    lr = 0.001
    batch_size = 16
    base_filters = 16

  ── [TRAIN] /usr/bin/python3 /content/Predictive-Maintenance-Audio-CNN/src/train.py --lr 0.001 --batch_size 16 --base_filters 16 --run_name run_lr1em03_bs16_bf16
  ────────────────────────────────────────────────────────
TRAINING — AudioClassifier on Log-Mel-Spectrograms
  Run name:     run_lr1em03_bs16_bf16
  Checkpoint:   /content/Predictive-Maintenance-Audio-CNN/models/best_model_run_lr1em03_bs16_bf16.pth

[1/5] Selecting compute device...
  Using device: CUDA (Tesla T4)

[2/5] Building DataLoaders...
BUILDING DATA PIPELINE

[1/4] Gathering file paths & labels...
  [normal] Found 3749 .npy files (label=0)
  [abnormal] Found 456 .npy files (label=1)

[2/4] Performing stratified train/val/test split...

  Split sizes (stratified, seed=42):
    train :  2943 total (2624 normal, 31

In [5]:
import shutil
from google.colab import files

# 1. Zip all the visualizations (including the new summary graph)
shutil.make_archive('tuning_visualizations', 'zip', '/content/Predictive-Maintenance-Audio-CNN/docs/visualizations')

# 2. Zip all the generated model weights
shutil.make_archive('tuning_models', 'zip', '/content/Predictive-Maintenance-Audio-CNN/models')

# 3. Download the zipped folders
files.download('tuning_visualizations.zip')
files.download('tuning_models.zip')

# 4. Download the master experiment tracker CSV
files.download('/content/Predictive-Maintenance-Audio-CNN/docs/experiment_tracking.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>